# Grammar Scoring Engine — Final E008 Submission

Frozen submission reproduction for SHL Research Engineer review. This notebook
verifies externally mounted artifacts, reports held-out development metrics,
and produces the final 216-row submission. It does not train or tune a model.


## Approach

Audio → canonical Whisper **large-v3** transcription → fine-tuned
**microsoft/deberta-v3-base** regression → five-fold inference → equal raw
prediction average.

E008 trains on the 732 examples with `label > 0`, excluding the anomalous
37-example zero-label block as an empirical distribution-mismatch experiment.
This does not establish that those labels are erroneous. Five frozen selected
checkpoints supply final inference; canonical transcripts are loaded unchanged.

Later experimental branches (E009, E012a, E013) were rejected and are not part of
final inference. There is no clipping, calibration, filename-ID rule, test
zero-block heuristic, or additional ensemble. Known public leaderboard result:
**0.3925** (reported separately from OOF development metrics).


## Environment and Paths

Attach the repository source, competition data, and frozen E008 artifact datasets
in Kaggle. Enable a GPU and Internet for missing runtime packages and the original
Hugging Face backbone/tokenizer downloads. A complete prepopulated Hugging Face
cache can supply these downloads offline without changing production inference.

Edit only the configuration cell below if discovery is ambiguous or the mounted
layout differs. Defaults recognize the repository artifact layout:
`models/E008/`, `transcripts/{train,test}.jsonl`, and
`oof/E008_deberta_no_zero_block.csv`. Individual artifacts may live in separate
mounts; override their variables below. Train transcripts are documented for
provenance and are not needed to infer test rows.

The repository is installed with its existing setuptools setup, without resolving
its development/visualization dependency minimums against Kaggle's preinstalled
stack. Only missing runtime dependencies are installed; existing packages are not
upgraded. Run cells in order in a clean Python 3.11+ session.


In [ ]:
import os
from pathlib import Path

# All Kaggle-specific paths and editable overrides live in this cell.
INPUT_DIR = Path("/kaggle/input")
REPO_DIR = None  # e.g. INPUT_DIR / "repository" / "grammar-scoring"
DATA_DIR = None  # directory containing train.csv and test.csv
ARTIFACT_DIR = None  # root containing models/E008 and/or canonical artifacts
E008_CHECKPOINT_DIR = None  # directory containing fold_0 through fold_4
TRAIN_TRANSCRIPT_PATH = None
TEST_TRANSCRIPT_PATH = None
E008_OOF_PATH = None
OUTPUT_DIR = Path("/kaggle/working")
SEED = 42  # Reproduction runtime seed; no fitting or model selection occurs.

# Discovery is deliberately strict: ambiguous matches require an override.
for variable, pattern, kind in (
    ("REPO_DIR", "pyproject.toml", "repo"),
    ("DATA_DIR", "test.csv", "data"),
    ("E008_CHECKPOINT_DIR", "fold_0/best.pt", "checkpoint"),
    ("TEST_TRANSCRIPT_PATH", "test.jsonl", "file"),
    ("E008_OOF_PATH", "E008_deberta_no_zero_block.csv", "file"),
):
    if globals()[variable] is not None:
        globals()[variable] = Path(globals()[variable])
        continue
    matches = []
    for candidate in INPUT_DIR.rglob(pattern):
        if kind == "repo":
            if not (
                candidate.parent / "src/grammar_scoring/inference/cli.py"
            ).is_file():
                continue
            candidate = candidate.parent
        elif kind == "data":
            if not (candidate.parent / "train.csv").is_file():
                continue
            candidate = candidate.parent
        elif kind == "checkpoint":
            candidate = candidate.parent.parent
            if candidate.name != "E008":
                continue
        matches.append(candidate)
    matches = sorted(set(matches))
    if len(matches) != 1:
        raise RuntimeError(
            f"Set {variable} in this configuration cell: found {len(matches)} "
            f"candidates under {INPUT_DIR}: {matches}"
        )
    globals()[variable] = matches[0]

if ARTIFACT_DIR is None:
    ARTIFACT_DIR = E008_CHECKPOINT_DIR.parent.parent
ARTIFACT_DIR = Path(ARTIFACT_DIR)
if TRAIN_TRANSCRIPT_PATH is None:
    TRAIN_TRANSCRIPT_PATH = TEST_TRANSCRIPT_PATH.with_name("train.jsonl")
TRAIN_TRANSCRIPT_PATH = Path(TRAIN_TRANSCRIPT_PATH)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
os.environ["GRAMMAR_DATA_DIR"] = str(DATA_DIR)
os.environ["GRAMMAR_ARTIFACT_DIR"] = str(ARTIFACT_DIR)
for variable in (
    "REPO_DIR",
    "DATA_DIR",
    "ARTIFACT_DIR",
    "E008_CHECKPOINT_DIR",
    "TRAIN_TRANSCRIPT_PATH",
    "TEST_TRANSCRIPT_PATH",
    "E008_OOF_PATH",
    "OUTPUT_DIR",
):
    print(f"{variable}: {globals()[variable]}")

In [ ]:
import importlib.util
import shutil
import subprocess
import sys

assert sys.version_info >= (3, 11), "The repository requires Python 3.11+"
# Build from a writable copy: Kaggle input mounts are read-only.

INSTALL_DIR = OUTPUT_DIR / "repository_runtime"
shutil.copytree(
    REPO_DIR,
    INSTALL_DIR,
    dirs_exist_ok=True,
    ignore=shutil.ignore_patterns(
        ".git", ".venv", "__pycache__", "artifacts", "data", ".pytest_cache"
    ),
)
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-deps",
        "--no-build-isolation",
        str(INSTALL_DIR),
    ],
    check=True,
)
missing = [
    package
    for module, package in (
        ("numpy", "numpy"),
        ("pandas", "pandas"),
        ("scipy", "scipy"),
        ("sklearn", "scikit-learn"),
        ("torch", "torch"),
        ("transformers", "transformers>=5,<6"),
        ("sentencepiece", "sentencepiece"),
    )
    if importlib.util.find_spec(module) is None
]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", *missing], check=True)

In [ ]:
import random

import numpy as np
import pandas as pd
import torch
import transformers
from IPython.display import display

from grammar_scoring.evaluation.metrics import regression_metrics
from grammar_scoring.inference.cli import load_test_inputs, validate_predictions
from grammar_scoring.inference.deberta import (
    discover_checkpoints,
    predict_deberta_ensemble,
)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Python: {sys.version.split()[0]}")
print(f"torch: {torch.__version__}; transformers: {transformers.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}; device used: {DEVICE}")

## Validate Competition Data

The competition `test.csv` is the row authority. Its placeholder labels are ignored.


In [ ]:
TRAIN_CSV = DATA_DIR / "train.csv"
TEST_CSV = DATA_DIR / "test.csv"
assert TRAIN_CSV.is_file(), f"Missing competition data: {TRAIN_CSV}"
assert TEST_CSV.is_file(), f"Missing competition data: {TEST_CSV}"
test = pd.read_csv(TEST_CSV, dtype={"filename": "string"})
assert "filename" in test.columns, "test.csv is missing filename"
assert len(test) == 216, f"Expected 216 test rows, found {len(test)}"
assert test["filename"].notna().all(), "Missing test filenames"
assert test["filename"].str.strip().ne("").all(), "Empty test filenames"
assert test["filename"].is_unique, "Duplicate test filenames"
print("Competition files present; 216 unique test filenames verified.")

## Validate E008 Artifacts

Production validators require exactly one canonical test transcript per expected
filename, no missing/unexpected records, and exactly the five canonical fold
checkpoint paths. Optional repository-generated `result.json` metadata binds each
checkpoint to its recorded SHA-256 and reports the original backbone revision.
These records are supplied by the external dataset, not an independent trust root.
Production E008 uses the original default Hugging Face loading behavior; recorded
revisions are reported without changing that behavior.


In [ ]:
import hashlib
import json

assert TEST_TRANSCRIPT_PATH.is_file(), f"Missing transcripts: {TEST_TRANSCRIPT_PATH}"
assert E008_OOF_PATH.is_file(), f"Missing frozen OOF artifact: {E008_OOF_PATH}"
frame = load_test_inputs(TEST_CSV, TEST_TRANSCRIPT_PATH)
assert len(frame) == 216
assert frame["filename"].tolist() == test["filename"].tolist()
checkpoints = discover_checkpoints(E008_CHECKPOINT_DIR, experiment="E008")
verified_hashes = 0
for fold, checkpoint in enumerate(checkpoints):
    metadata_path = checkpoint.with_name("result.json")
    if not metadata_path.is_file():
        print(f"fold_{fold}: best.pt present; no result.json provenance metadata")
        continue
    metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    assert metadata["fold"] == fold, f"Fold identity mismatch: {metadata_path}"
    model_metadata = metadata.get("model", {})
    if model_metadata.get("name"):
        assert model_metadata["name"] == "microsoft/deberta-v3-base"
    expected_digest = metadata.get("checkpoint_sha256")
    if expected_digest:
        digest = hashlib.sha256()
        with checkpoint.open("rb") as stream:
            for block in iter(lambda: stream.read(1024 * 1024), b""):
                digest.update(block)
        assert digest.hexdigest() == expected_digest, f"Checksum mismatch: {checkpoint}"
        verified_hashes += 1
    print(
        f"fold_{fold}: hash verified={bool(expected_digest)}; "
        f"recorded backbone revision={model_metadata.get('revision')}"
    )
print(
    f"Structural verification passed; checkpoint hashes verified: {verified_hashes}/5"
)
if verified_hashes < 5:
    print("Complete selected artifact checksums are unavailable; no hashes invented.")
print("Provenance depends on the externally mounted frozen E008 artifact dataset.")

## Development Metrics

Recompute **OOF development RMSE** and Pearson from frozen predictions on the
common 732-row retained population. These are held-out development metrics, not
test RMSE. The reference constants below are integrity checks, not metric outputs.


In [ ]:
oof = pd.read_csv(E008_OOF_PATH, dtype={"filename": "string"})
assert {"filename", "label", "fold", "prediction"} <= set(oof.columns)
assert len(oof) == 732, f"Expected 732 OOF rows, found {len(oof)}"
assert oof["filename"].notna().all() and oof["filename"].str.strip().ne("").all()
assert oof["filename"].is_unique, "OOF filenames must be unique"
assert oof["label"].gt(0).all(), "OOF population must have label > 0"
assert oof["fold"].notna().all() and set(oof["fold"]) == set(range(5))
assert np.isfinite(oof[["label", "prediction"]].to_numpy(dtype=float)).all()
metrics = regression_metrics(oof["label"].to_numpy(), oof["prediction"].to_numpy())
np.testing.assert_allclose(metrics["rmse"], 0.6197891638194158, rtol=0, atol=1e-10)
np.testing.assert_allclose(
    metrics["pearson_correlation"], 0.7942308488890498, rtol=0, atol=1e-10
)
print(f"OOF development RMSE (n=732): {metrics['rmse']:.16f}")
print(f"OOF development Pearson: {metrics['pearson_correlation']:.16f}")

## Run Frozen E008 Inference

Call the audited production implementation. It loads five checkpoints and returns
their equal raw average in competition test order. No training or postprocessing
is performed. CPU inference is supported but slower than GPU inference.


In [ ]:
predictions = predict_deberta_ensemble(
    frame, E008_CHECKPOINT_DIR, device=DEVICE, experiment="E008"
)

## Validate Predictions

Statistics are diagnostic only; predictions remain unchanged.


In [ ]:
validate_predictions(predictions, test)
assert len(predictions) == 216
labels = predictions["label"].to_numpy(dtype=np.float64)
print(f"Count: {len(labels)}")
print(f"Mean: {labels.mean():.12f}")
print(f"Population std: {labels.std(ddof=0):.12f}")
print(f"Min: {labels.min():.12f}")
print(f"Median: {np.median(labels):.12f}")
print(f"Max: {labels.max():.12f}")

## Create submission.csv

Write the raw production predictions and verify the serialized file again.


In [ ]:
SUBMISSION_PATH = OUTPUT_DIR / "submission.csv"
predictions.to_csv(SUBMISSION_PATH, index=False)
submission = pd.read_csv(
    SUBMISSION_PATH, dtype={"filename": "string"}, float_precision="round_trip"
)
validate_predictions(submission, test)
assert len(submission) == 216
assert submission.columns.tolist() == ["filename", "label"]
assert submission["filename"].is_unique
assert submission["filename"].tolist() == test["filename"].tolist()
assert np.isfinite(submission["label"].to_numpy(dtype=np.float64)).all()
np.testing.assert_array_equal(submission["label"].to_numpy(), labels)
display(submission.head(5))

## Final Summary

E008 is the final model. Development metrics were recomputed from the frozen OOF
artifact and checked against the selected result. Final predictions are the raw
equal average of five frozen checkpoints, with verified test identities and order.

After all cells succeed, submit `/kaggle/working/submission.csv` to the competition.
The known public leaderboard result **0.3925** is historical, not recomputed here.
Save this notebook with execution counts and outputs cleared when committing it;
competition data, predictions, credentials, and checkpoints must remain external.


In [ ]:
print(f"Selected model: E008; device: {DEVICE}; runtime seed: {SEED}")
print(f"OOF development RMSE: {metrics['rmse']:.16f}")
print(f"OOF development Pearson: {metrics['pearson_correlation']:.16f}")
print(f"Validated submission: {SUBMISSION_PATH} ({len(submission)} rows)")